# 1. Profile the bag to get metadata + topics

In [8]:
from pathlib import Path
from hephaes_core.profiler import Profiler

bag_files = [
    # 'input/ros1.bag',
    # 'input/ros2.mcap',
    # 'input/sample.mcap',
    'input/sample2.mcap',
    ]
topics = []

if not bag_files:
    print("No bag files found in", input_dir)
else:
    prof = Profiler(bag_files, max_workers=1)
    metadata_list = prof.profile()
    for meta in metadata_list:
        print(f"File: {meta.file_path}")
        print(f"Path: {meta.path}")
        print(f"ROS version: {meta.ros_version}")
        print(f"Storage format: {meta.storage_format}")
        print(f"File size (bytes): {meta.file_size_bytes}")
        print(f"Start: {meta.start_time_iso} End: {meta.end_time_iso} Duration(s): {meta.duration_seconds}")
        print(f"Message count: {meta.message_count}")
        print("Topics:")
        topics = meta.topics
        for t in meta.topics:
            print(f"  - {t.name} ({t.message_type}): {t.message_count} messages, {t.rate_hz} Hz")
        print("-" * 40)




File: input/sample2.mcap
Path: input/sample2.mcap
ROS version: ROS2
Storage format: mcap
File size (bytes): 1303247009
Start: 2020-01-01T01:03:00.699778Z End: 2020-01-01T01:05:42.897412Z Duration(s): 162.197633983
Message count: 37701
Topics:
  - /left_camera/image/compressed (sensor_msgs/msg/CompressedImage): 1622 messages, 10.01 Hz
  - /livox/imu (sensor_msgs/msg/Imu): 32835 messages, 202.44 Hz
  - /livox/lidar (livox_ros_driver/msg/CustomMsg): 1622 messages, 10.01 Hz
  - /right_camera/image/compressed (sensor_msgs/msg/CompressedImage): 1622 messages, 10.01 Hz
----------------------------------------


# 2. Create a MappingTemplate

In [9]:
from hephaes_core.mappers import build_mapping_template

mapping = build_mapping_template(topics)

# 3. Use Converter to convert

In [10]:
from hephaes_core import Converter, ResampleConfig

# Default: no resampling (union timestamps, null when a column has no value).
parquet_files = Converter(bag_files, mapping, "./output").convert()

# Optional: regular-grid downsample or interpolate.
# resample_cfg = ResampleConfig(freq_hz=10.0, method="downsample")
# parquet_files = Converter(bag_files, mapping, "./output", resample=resample_cfg).convert()


# 4. Stream the first few rows of the output

In [11]:
from hephaes_core.parquet import stream_wide_parquet_rows

output_parquet = parquet_files[0]
n_rows = 5

for i, row in enumerate(stream_wide_parquet_rows(output_parquet)):
    print(row)
    if i + 1 >= n_rows:
        break


{'episode_id': 'episode_0001', 'bag_path': 'input/sample2.mcap', 'ros_version': 'ROS2', 'timestamp_ns': 1577840580699778079, 'left_camera_image_compressed': '{"__bytes__":true,"encoding":"base64","value":"..."}', 'livox_imu': None, 'livox_lidar': None, 'right_camera_image_compressed': '{"__bytes__":true,"encoding":"base64","value":"..."}'}
{'episode_id': 'episode_0001', 'bag_path': 'input/sample2.mcap', 'ros_version': 'ROS2', 'timestamp_ns': 1577840580699778080, 'left_camera_image_compressed': None, 'livox_imu': None, 'livox_lidar': '{"__bytes__":true,"encoding":"base64","value":"..."}', 'right_camera_image_compressed': None}
{'episode_id': 'episode_0001', 'bag_path': 'input/sample2.mcap', 'ros_version': 'ROS2', 'timestamp_ns': 1577840580702645063, 'left_camera_image_compressed': None, 'livox_imu': '{"__bytes__":true,"encoding":"base64","value":"AAEAAMTvC15Hg+EpDAAAAGxpdm94X2ZyYW1lAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA